# 🟢 **Generate Test**

## 🔴 **Imports**

In [77]:
import os
import math
import re
from pathlib import Path
import statistics
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer

## 🔴 **CONFIG**

In [78]:
MODEL_PATH = os.path.join(
    "ark-alpha-mini-v1-2.9M.pt",
)

TOKENIZER_PATH = os.path.join(
    "tokenizer",
    "bpe-tokenizer_alphabet_nlp_dataset_10,000.json",
)


TEST_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

## 🔴 **START MODEL TEST**

In [47]:
print("=" * 100)
print("🧪 INDEPENDENT MODEL TEST")
print("=" * 100)

print(f"Device: {TEST_DEVICE}")
print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Model: {MODEL_PATH}")

🧪 INDEPENDENT MODEL TEST
Device: cpu
Tokenizer: tokenizer\bpe-tokenizer_alphabet_nlp_dataset_10,000.json
Model: ark-alpha-mini-v1-2.9M.pt


## 🔴 **FILE CHECK**

In [48]:
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(
        f"❌ Tokenizer not found:\n{TOKENIZER_PATH}"
    )

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"❌ Model not found:\n{MODEL_PATH}"
    )

print("✅ Tokenizer file found.")
print("✅ Model file found.")

✅ Tokenizer file found.
✅ Model file found.


## 🔴 **LOAD TOKENIZER**

In [49]:
test_tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print("✅ Tokenizer loaded successfully.")
print(
    f"Vocabulary size: "
    f"{test_tokenizer.get_vocab_size():,}"
)

✅ Tokenizer loaded successfully.
Vocabulary size: 10,000


## 🔴 **MULTI-HEAD ATTENTION**

In [50]:
class IndependentMultiHeadAttention(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        dropout_rate
    ):
        super().__init__()

        if n_embd % n_head != 0:
            raise ValueError(
                "n_embd must be divisible by n_head."
            )

        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head

        self.qkv_proj = nn.Linear(
            n_embd,
            3 * n_embd,
            bias=False
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.c_proj.residual = True

    def forward(self, x):

        B, T, C = x.shape

        qkv = (
            self.qkv_proj(x)
            .view(
                B,
                T,
                3,
                self.n_head,
                self.head_size
            )
            .permute(
                2,
                0,
                3,
                1,
                4
            )
        )

        q, k, v = qkv.unbind(0)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                C
            )
        )

        return self.c_proj(y)

## 🔴 **FEED FORWARD**

In [51]:
class IndependentFeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        hidden_size = int(
            f_expnd * n_embd
        )

        self.up_proj = nn.Linear(
            n_embd,
            hidden_size,
            bias=False
        )

        self.down_proj = nn.Linear(
            hidden_size,
            n_embd,
            bias=False
        )

        self.down_proj.residual = True

        self.mlp_dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        return self.mlp_dropout(
            self.down_proj(
                F.gelu(
                    self.up_proj(x)
                )
            )
        )

## 🔴 **DECODER BLOCK**

In [52]:
class IndependentDecoderBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.mha = IndependentMultiHeadAttention(
            n_embd,
            n_head,
            dropout_rate
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

        self.mlp = IndependentFeedForward(
            n_embd,
            f_expnd,
            dropout_rate
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        x = x + self.dropout(
            self.mha(
                self.ln1(x)
            )
        )

        x = x + self.dropout(
            self.mlp(
                self.ln2(x)
            )
        )

        return x

## 🔴 **ARK MODEL**

In [53]:
class IndependentARK(nn.Module):

    def __init__(
        self,
        vocab_size=10000,
        max_seq_len=1024,
        n_layer=8,
        n_head=16,
        n_embd=128,
        f_expnd=4,
        dropout_rate=0.2
    ):
        super().__init__()

        self.wte = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.wpe = nn.Embedding(
            max_seq_len,
            n_embd
        )

        self.decoders = nn.ModuleList(
            [
                IndependentDecoderBlock(
                    n_embd=n_embd,
                    n_head=n_head,
                    f_expnd=f_expnd,
                    dropout_rate=dropout_rate
                )
                for _ in range(n_layer)
            ]
        )

        self.lnf = nn.LayerNorm(
            n_embd
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = self.wte.weight

    def forward(self, idx):

        B, T = idx.shape

        if T > self.wpe.num_embeddings:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.wpe.num_embeddings}"
            )

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = (
            self.wte(idx)
            +
            self.wpe(positions)
        )

        for decoder in self.decoders:
            x = decoder(x)

        x = self.lnf(x)

        return self.lm_head(x)

## 🔴 **CREATE MODEL**

In [54]:
test_model = IndependentARK(
    vocab_size=10_000,
    max_seq_len=128,
    n_layer=8,
    n_head=16,
    n_embd=128,
    f_expnd=4,
    dropout_rate=0.2
).to(TEST_DEVICE)

## 🔴 **PARAMETER COUNT**

In [55]:
total_params = sum(
    p.numel()
    for p in test_model.parameters()
)

print(
    f"📊 Parameters: "
    f"{total_params:,} "
    f"({total_params / 1e6:.2f}M)"
)

📊 Parameters: 2,873,600 (2.87M)


## 🔴 **LOAD CHECKPOINT**

In [56]:
print(
    "🔄 Loading model.pt..."
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=TEST_DEVICE
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        "❌ Unexpected checkpoint format."
    )

print("\n🔍 Checkpoint information")

print("Checkpoint keys:")
for key in checkpoint.keys():
    print(f"   • {key}")

# Display training progress information
if "optimizer_step" in checkpoint:
    print(
        f"\n📈 Optimizer step: "
        f"{checkpoint['optimizer_step']:,}"
    )
else:
    print(
        "\n⚠️ 'optimizer_step' not found in checkpoint."
    )

if "seen_tokens" in checkpoint:
    print(
        f"📊 Seen tokens: "
        f"{checkpoint['seen_tokens']:,}"
    )
else:
    print(
        "⚠️ 'seen_tokens' not found in checkpoint."
    )

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "❌ 'model_state_dict' not found in checkpoint."
    )

test_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_model.eval()

print(
    "\n✅ model.pt loaded successfully."
)

🔄 Loading model.pt...

🔍 Checkpoint information
Checkpoint keys:
   • model_state_dict
   • optimizer_state_dict
   • optimizer_step
   • seen_tokens

📈 Optimizer step: 21,668
📊 Seen tokens: 11,097,184

✅ model.pt loaded successfully.


## 🔴 **CHECKPOINT VERIFICATION**

In [57]:
checkpoint_state = checkpoint[
    "model_state_dict"
]

model_state = test_model.state_dict()

checkpoint_keys = set(
    checkpoint_state.keys()
)

model_keys = set(
    model_state.keys()
)

missing_keys = (
    model_keys -
    checkpoint_keys
)

unexpected_keys = (
    checkpoint_keys -
    model_keys
)

shape_mismatches = []

for key in checkpoint_keys & model_keys:

    if (
        checkpoint_state[key].shape
        !=
        model_state[key].shape
    ):
        shape_mismatches.append(
            (
                key,
                checkpoint_state[key].shape,
                model_state[key].shape
            )
        )

print(
    "🔍 Checkpoint verification"
)

print(
    f"Checkpoint tensors: "
    f"{len(checkpoint_keys):,}"
)

print(
    f"Model tensors:      "
    f"{len(model_keys):,}"
)

if missing_keys:
    print("\n❌ Missing keys:")
    for key in sorted(missing_keys):
        print("   ", key)
else:
    print("✅ No missing keys.")

if unexpected_keys:
    print("\n⚠️ Unexpected keys:")
    for key in sorted(unexpected_keys):
        print("   ", key)
else:
    print("✅ No unexpected keys.")

if shape_mismatches:
    print("\n❌ Shape mismatches:")
    for (
        key,
        checkpoint_shape,
        model_shape
    ) in shape_mismatches:
        print(
            f"   {key}: "
            f"checkpoint={checkpoint_shape}, "
            f"model={model_shape}"
        )
else:
    print("✅ All tensor shapes match.")

if (
    not missing_keys
    and not unexpected_keys
    and not shape_mismatches
):
    print(
        "\n🎯 Checkpoint architecture "
        "matches the model perfectly."
    )
else:
    raise RuntimeError(
        "\n❌ Checkpoint is not fully compatible."
    )

🔍 Checkpoint verification
Checkpoint tensors: 69
Model tensors:      69
✅ No missing keys.
✅ No unexpected keys.
✅ All tensor shapes match.

🎯 Checkpoint architecture matches the model perfectly.


## 🔴 **GENERATION FUNCTION**

In [58]:
def independent_generate(
    model,
    tokenizer,
    prompt,
    max_seq_len=128,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device="cuda",
    seed=42
):

    model.eval()

    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    if len(prompt_ids) == 0:
        raise ValueError(
            "❌ Prompt produced zero tokens."
        )

    if len(prompt_ids) >= max_seq_len:
        raise ValueError(
            "❌ Prompt is already equal to "
            "or longer than max_seq_len."
        )

    inputs = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    rng = torch.Generator(
        device=device
    )

    rng.manual_seed(seed)

    with torch.no_grad():

        while inputs.shape[1] < max_seq_len:

            logits = model(
                inputs
            )

            next_logits = logits[
                :,
                -1,
                :
            ]

            if greedy:

                next_token = torch.argmax(
                    next_logits,
                    dim=-1
                )

            else:

                temperature = max(
                    temperature,
                    1e-5
                )

                probs = torch.softmax(
                    next_logits / temperature,
                    dim=-1
                )

                k = min(
                    top_k,
                    probs.shape[-1]
                )

                topk_probs, topk_indices = (
                    torch.topk(
                        probs,
                        k=k,
                        dim=-1
                    )
                )

                # Important:
                # torch.multinomial expects a valid probability
                # distribution. Re-normalize top-k probabilities.
                topk_probs = (
                    topk_probs
                    /
                    topk_probs.sum(
                        dim=-1,
                        keepdim=True
                    )
                )

                sampled = torch.multinomial(
                    topk_probs,
                    num_samples=1,
                    generator=rng
                )

                next_token = torch.gather(
                    topk_indices,
                    -1,
                    sampled
                ).squeeze(-1)

            inputs = torch.cat(
                [
                    inputs,
                    next_token.unsqueeze(-1)
                ],
                dim=-1
            )

    generated_ids = inputs[
        0,
        len(prompt_ids):
    ].tolist()

    return tokenizer.decode(
        generated_ids
    )

## 🔴 **TEST PROMPT**

In [59]:
TEST_PROMPT = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)

## 🔴 **NEXT TOKEN PROBABILITY TEST**

In [60]:
print("=" * 100)
print("🔬 NEXT TOKEN PROBABILITY TEST")
print("=" * 100)

prompt = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)

print("\n[Prompt]")
print(prompt)

ids = test_tokenizer.encode(
    prompt
).ids

print("\n[Prompt Token IDs]")
print(ids)

print(
    f"\nPrompt token count: "
    f"{len(ids)}"
)

x = torch.tensor(
    [ids],
    dtype=torch.long,
    device=TEST_DEVICE
)

with torch.no_grad():
    logits = test_model(x)

next_logits = logits[
    :,
    -1,
    :
]

probs = torch.softmax(
    next_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probs,
    k=min(20, probs.shape[-1]),
    dim=-1
)

print("\n")
print("-" * 100)
print("TOP-20 NEXT TOKEN PREDICTIONS")
print("-" * 100)
print(
    f"{'Rank':>4} "
    f"{'ID':>6} "
    f"{'Probability':>14} "
    f"{'Token':>30}"
)
print("-" * 100)

for rank, (p, idx) in enumerate(
    zip(
        top_probs[0],
        top_ids[0]
    ),
    start=1
):
    token = test_tokenizer.decode(
        [idx.item()]
    )

    print(
        f"{rank:4d} "
        f"{idx.item():6d} "
        f"{p.item():14.8f} "
        f"{repr(token):>30}"
    )

🔬 NEXT TOKEN PROBABILITY TEST

[Prompt]
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که

[Prompt Token IDs]
[681, 806, 265, 5041, 3120, 362, 792, 45, 614, 652, 222, 490, 9557, 1409, 151, 219, 464, 534, 221, 237]

Prompt token count: 20


----------------------------------------------------------------------------------------------------
TOP-20 NEXT TOKEN PREDICTIONS
----------------------------------------------------------------------------------------------------
Rank     ID    Probability                          Token
----------------------------------------------------------------------------------------------------
   1    213     0.09790432                          ' در'
   2    212     0.09301607                          ' به'
   3    275     0.05749801                        ' برای'
   4    210     0.05011905                          ' می'
   5    264     0.04346023                          ' با'
   6    222     0.03057924                          '

## 🔴 **EXPECTED CONTINUATION TEST**

In [61]:
print("=" * 100)
print("🎯 EXPECTED NEXT TOKEN TEST")
print("=" * 100)

expected = (
    " به سیستم‌ها اجازه می‌دهد"
)

expected_ids = test_tokenizer.encode(
    expected
).ids

print("\n[Expected continuation]")
print(repr(expected))

print("\n[Expected token IDs]")
print(expected_ids)

print("\n[Expected tokens]")

for position, idx in enumerate(
    expected_ids,
    start=1
):
    token = test_tokenizer.decode(
        [idx]
    )

    print(
        f"{position:3d}. "
        f"ID={idx:5d} "
        f"Token={repr(token)}"
    )

if len(expected_ids) > 0:

    expected_next_id = expected_ids[0]

    expected_prob = (
        probs[
            0,
            expected_next_id
        ].item()
    )

    print("\n")
    print("-" * 100)
    print("ACTUAL EXPECTED NEXT TOKEN")
    print("-" * 100)

    print(
        "Expected token ID:",
        expected_next_id
    )

    print(
        "Expected token:",
        repr(
            test_tokenizer.decode(
                [expected_next_id]
            )
        )
    )

    print(
        "Probability:",
        f"{expected_prob:.10f}"
    )

    sorted_ids = torch.argsort(
        probs[0],
        descending=True
    )

    expected_rank_tensor = torch.where(
        sorted_ids == expected_next_id
    )[0]

    if len(expected_rank_tensor) > 0:
        expected_rank = (
            expected_rank_tensor[0].item()
            + 1
        )

        print(
            "Rank:",
            expected_rank
        )
    else:
        print("Rank: Not found")

🎯 EXPECTED NEXT TOKEN TEST

[Expected continuation]
' به سیستم\u200cها اجازه می\u200cدهد'

[Expected token IDs]
[212, 412, 1409, 151, 239, 1734, 210, 1409, 151, 511]

[Expected tokens]
  1. ID=  212 Token=' به'
  2. ID=  412 Token=' سیستم'
  3. ID= 1409 Token='�'
  4. ID=  151 Token='�'
  5. ID=  239 Token='ها'
  6. ID= 1734 Token=' اجازه'
  7. ID=  210 Token=' می'
  8. ID= 1409 Token='�'
  9. ID=  151 Token='�'
 10. ID=  511 Token='دهد'


----------------------------------------------------------------------------------------------------
ACTUAL EXPECTED NEXT TOKEN
----------------------------------------------------------------------------------------------------
Expected token ID: 212
Expected token: ' به'
Probability: 0.0930160657
Rank: 2


## 🔴 **GREEDY DECODING**

In [62]:
print("=" * 100)
print("🧠 GREEDY DECODING")
print("=" * 100)

greedy_output = independent_generate(
    model=test_model,
    tokenizer=test_tokenizer,
    prompt=TEST_PROMPT,
    max_seq_len=128,
    greedy=True,
    device=TEST_DEVICE
)

print("\n[Prompt]")
print(TEST_PROMPT)

print("\n[Generated continuation]")
print(greedy_output)

🧠 GREEDY DECODING

[Prompt]
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که

[Generated continuation]
 در این‌ها به‌صورت زیر است که به‌صورت برنامه‌های عصبی مصنوعی، به‌های عصبی، به‌های عصبی، مدل‌های عصبی مصنوعی، به‌های عصبی، به‌صورت `C` استفاده از یک‌های عصبی مصنوعی، یک برنامه‌های عصبی، به‌های عصبی مصنوعی، برنامه‌های عصبی، از آن‌های عصبی مصنوعی، برنامه‌های عصبی مصنوعی، از یک‌های عصبی، استفاده از


## 🔴 **TOP-K SAMPLING**

In [63]:
print("=" * 100)
print("🎲 TOP-K SAMPLING")
print("=" * 100)

for sample_number in range(
    1,
    4
):

    output = independent_generate(
        model=test_model,
        tokenizer=test_tokenizer,
        prompt=TEST_PROMPT,
        max_seq_len=128,
        temperature=0.7,
        top_k=10,
        greedy=False,
        device=TEST_DEVICE,
        seed=42 + sample_number
    )

    print(
        f"\n[Sample {sample_number}]"
    )

    print(output)

🎲 TOP-K SAMPLING

[Sample 1]
 برای داده‌های برنامه‌نویسی، در یک داده‌های برنامه‌نویسی برنامه‌های مختلف و به مدل‌های عصبی عمیق‌های وب و یادگیری ماشین و استفاده از این کارِ برنامه‌نویسی است که برای استفاده از آن‌های هوش مصنوعی و به‌نویسی عصبی مصنوعی، مدل و توسعه‌های اصلی، یک‌ها برای استفاده از سیستم‌نویسی برنامه‌ها (C-.

-- در آن، مدل‌های وب

[Sample 2]
 برای استفاده از آن‌ها استفاده می‌شود.

- **ه‌شود که‌شوند:**
- **م‌توان **تفسیر:**
**مثال:**
- **ح‌تواند با مثال، `A, 2. **م‌توان به‌های زبانی زبانی**تفسیر:**
- **حم‌شود:**
- **بِسَرُسْمُسحَاً به‌های مختلف:**
- **این‌

[Sample 3]
 به‌صورت زیر است که به‌عنوان یک‌صورت `C` و مدل‌صورت برنامه‌صورت زیر است که از آن، مدل‌های بزرگ و در این است، مدل‌ها استفاده از داده‌های عصبی مصنوعی برای توسعه‌های مختلف استفاده از آن‌های مختلف، می‌ها می‌های برنامه‌های وب‌های برنامه‌ی عصبی مصنوعی، برای استفاده از برنامه‌های بزرگ، به‌های برنامه‌


## 🔴 **TOKENIZER TEST**

In [64]:
print("=" * 100)
print("✅ TOKENIZER TEST")
print("=" * 100)

texts = [
    "یادگیری ماشین",
    (
        "یادگیری ماشین "
        "(Machine Learning - ML)، "
        "یکی از زیرشاخه‌های هوش مصنوعی است که"
    )
]

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    print("\nTEXT:")
    print(text)

    print("IDS:")
    print(enc.ids)

    print("TOKENS:")
    print(enc.tokens)

✅ TOKENIZER TEST

TEXT:
یادگیری ماشین
IDS:
[681, 806]
TOKENS:
['ĠÛĮØ§Ø¯Ú¯ÛĮØ±ÛĮ', 'ĠÙħØ§Ø´ÛĮÙĨ']

TEXT:
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که
IDS:
[681, 806, 265, 5041, 3120, 362, 792, 45, 614, 652, 222, 490, 9557, 1409, 151, 219, 464, 534, 221, 237]
TOKENS:
['ĠÛĮØ§Ø¯Ú¯ÛĮØ±ÛĮ', 'ĠÙħØ§Ø´ÛĮÙĨ', 'Ġ(', 'Machine', 'ĠLearning', 'Ġ-', 'ĠM', 'L', ')ØĮ', 'ĠÛĮÚ©ÛĮ', 'ĠØ§Ø²', 'ĠØ²ÛĮØ±', 'Ø´Ø§Ø®Ùĩ', 'âĢ', 'Į', 'ÙĩØ§ÛĮ', 'ĠÙĩÙĪØ´', 'ĠÙħØµÙĨÙĪØ¹ÛĮ', 'ĠØ§Ø³Øª', 'ĠÚ©Ùĩ']


## 🔴 **TOKENIZER ROUND-TRIP TEST**

In [65]:
print("=" * 80)
print("🔬 TOKENIZER ROUND-TRIP TEST")
print("=" * 80)

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    decoded = test_tokenizer.decode(
        enc.ids
    )

    print("\nOriginal:")
    print(repr(text))

    print("Decoded:")
    print(repr(decoded))

    print(
        "MATCH:",
        text == decoded
    )

🔬 TOKENIZER ROUND-TRIP TEST

Original:
'یادگیری ماشین'
Decoded:
' یادگیری ماشین'
MATCH: False

Original:
'یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که'
Decoded:
' یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که'
MATCH: False


## 🔴 **FULL TARGET TEST**

In [66]:
target = """یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که به سیستم‌ها اجازه می‌دهد بدون برنامه‌ریزی صریح، از داده‌ها یاد بگیرند و بهبود یابند.

الگوریتم‌های یادگیری ماشین به سه دسته‌ی اصلی تقسیم می‌شوند:

- **یادگیری نظارت‌شده (Supervised Learning)**
- **یادگیری بدون نظارت (Unsupervised Learning)**
- **یادگیری تقویتی (Reinforcement Learning)**"""

enc = test_tokenizer.encode(
    target
)

decoded = test_tokenizer.decode(
    enc.ids
)

print("=" * 100)
print("📚 FULL TARGET TOKENIZER TEST")
print("=" * 100)

print("\nOriginal:")
print(repr(target))

print("\nDecoded:")
print(repr(decoded))

print(
    "\nMATCH:",
    target == decoded
)

print(
    "Token count:",
    len(enc.ids)
)

📚 FULL TARGET TOKENIZER TEST

Original:
'یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که به سیستم\u200cها اجازه می\u200cدهد بدون برنامه\u200cریزی صریح، از داده\u200cها یاد بگیرند و بهبود یابند.\n\nالگوریتم\u200cهای یادگیری ماشین به سه دسته\u200cی اصلی تقسیم می\u200cشوند:\n\n- **یادگیری نظارت\u200cشده (Supervised Learning)**\n- **یادگیری بدون نظارت (Unsupervised Learning)**\n- **یادگیری تقویتی (Reinforcement Learning)**'

Decoded:
' یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که به سیستم\u200cها اجازه می\u200cدهد بدون برنامه\u200cریزی صریح، از داده\u200cها یاد بگیرند و بهبود یابند.\n\nالگوریتم\u200cهای یادگیری ماشین به سه دسته\u200cی اصلی تقسیم می\u200cشوند:\n\n- **یادگیری نظارت\u200cشده (Supervised Learning)**\n- **یادگیری بدون نظارت (Unsupervised Learning)**\n- **یادگیری تقویتی (Reinforcement Learning)**'

MATCH: False
Token count: 104


## 🔴 **TOKENIZER LONG TEST**

In [74]:
target = """هوش مصنوعی یکی از مهم‌ترین شاخه‌های علوم کامپیوتر است که هدف آن ساخت سیستم‌هایی است که بتوانند وظایفی مانند یادگیری، استدلال، تشخیص الگو، پردازش زبان طبیعی و تصمیم‌گیری را انجام دهند. یادگیری ماشین یا Machine Learning زیرمجموعه‌ای از هوش مصنوعی است که در آن مدل به جای دریافت مجموعه‌ای از قوانین ثابت، الگوها را از داده‌ها یاد می‌گیرد.

برای مثال، یک مدل زبانی می‌تواند جمله‌ای مانند «امروز هوا بسیار خوب است» را به مجموعه‌ای از tokenها تبدیل کند. سپس هر token به یک شناسه عددی تبدیل می‌شود و مدل این شناسه‌ها را به embedding vector تبدیل می‌کند. اگر vocabulary شامل 10,000 token باشد، هر token یک شناسه در بازه 0 تا 9,999 خواهد داشت.

یکی از معماری‌های معروف برای مدل‌های زبانی Transformer است. در این معماری، مکانیزم Attention به مدل اجازه می‌دهد ارتباط میان بخش‌های مختلف یک توالی را بررسی کند. برای مثال، در جمله «علی کتابی را که دیروز خریده بود خواند»، مدل باید بتواند رابطه میان «کتابی» و «خریده بود» را تا حدی درک کند.

فرمول ساده Attention به صورت زیر نوشته می‌شود:

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

در این رابطه Q یا Query، K یا Key و V یا Value هستند و d_k ابعاد بردار Key است. تابع softmax مقادیر را به یک توزیع احتمال تبدیل می‌کند.

در ریاضیات، اگر x = 5 و y = 3 باشد، آنگاه x + y = 8 و x × y = 15 است. همچنین داریم:

a^2 + b^2 = c^2

که همان قضیه فیثاغورس است. مشتق تابع f(x) = x^2 برابر است با:

f'(x) = 2x

و انتگرال آن به شکل زیر نوشته می‌شود:

∫ x^2 dx = x^3/3 + C

یک مثال دیگر از یک دنباله ریاضی:

2, 4, 8, 16, 32, 64, 128, ...

در این دنباله هر عدد دو برابر عدد قبلی است. بنابراین جمله nام را می‌توان به صورت a_n = 2^n نوشت، البته با توجه به شماره‌گذاری جمله اول.

در فیزیک، قانون دوم نیوتن به صورت F = ma بیان می‌شود. اگر جرم جسمی 10 kg و شتاب آن 2 m/s² باشد، نیروی وارد بر جسم برابر با 20 N خواهد بود. انرژی جنبشی یک جسم نیز از رابطه زیر محاسبه می‌شود:

E_k = 1/2 mv^2

در نسبیت خاص، رابطه معروف E = mc^2 میان جرم و انرژی ارتباط برقرار می‌کند. اگر سرعت نور را تقریباً برابر با 3 × 10^8 m/s در نظر بگیریم، مقدار c^2 برابر با 9 × 10^16 m²/s² خواهد بود.

در علوم کامپیوتر، الگوریتم‌ها مجموعه‌ای از دستورالعمل‌ها برای حل مسئله هستند. پیچیدگی زمانی الگوریتم جستجوی دودویی معمولاً O(log n) است، در حالی که جستجوی خطی در بدترین حالت O(n) عملیات انجام می‌دهد. ساختارهای داده‌ای مانند array، linked list، stack، queue، hash table، tree و graph در طراحی نرم‌افزار بسیار مهم هستند.

یک قطعه کد Python می‌تواند به شکل زیر باشد:

def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

for i in range(10):
    print(fibonacci(i))

در Python، متغیرها می‌توانند انواع مختلفی داشته باشند:

name = "ARK"
version = 1.0
parameters = 2990000
learning_rate = 3e-4
is_training = True

یک مدل PyTorch ممکن است به صورت زیر تعریف شود:

import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.linear = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        return self.linear(x)

model = SimpleModel(vocab_size=10000, n_embd=64)

در آموزش یک مدل زبانی، معمولاً یک توالی از tokenها به مدل داده می‌شود و مدل تلاش می‌کند token بعدی را پیش‌بینی کند. برای مثال اگر ورودی «هوش مصنوعی در حال» باشد، مدل باید احتمال tokenهای مختلف را برای ادامه جمله محاسبه کند. اگر توزیع احتمالات مدل برای کلمات مختلف به صورت زیر باشد:

آینده: 0.05
یادگیری: 0.10
توسعه: 0.15
رشد: 0.05
پیشرفت: 0.20
فعالیت: 0.03
است: 0.07

مدل در نهایت یک توزیع احتمال روی کل vocabulary تولید می‌کند. در یک مسئله classification یا language modeling، Cross-Entropy Loss یکی از معیارهای رایج برای اندازه‌گیری اختلاف میان پیش‌بینی مدل و target واقعی است.

اگر احتمال اختصاص داده‌شده به token صحیح برابر p باشد، loss مربوط به آن نمونه را می‌توان به صورت زیر نوشت:

L = -log(p)

بنابراین اگر مدل احتمال 0.9 را به token صحیح بدهد، loss کوچک خواهد بود، اما اگر احتمال token صحیح فقط 0.01 باشد، loss بسیار بزرگ‌تر خواهد شد.

در زبان انگلیسی، جمله‌های زیر نمونه‌هایی از ساختارهای متفاوت هستند:

The model is learning from data.
Artificial intelligence can process natural language.
Transformers use attention mechanisms.
The quick brown fox jumps over the lazy dog.
Machine learning is based on mathematical optimization.
What happens when the model receives a new sentence?
The answer depends on the learned parameters.

Some technical terms should also be tested:

tokenization
tokenizer
vocabulary
embedding
attention
transformer
neural network
backpropagation
gradient descent
optimization
probability
distribution
classification
regression
generalization
overfitting
underfitting
fine-tuning
pretraining
inference
context window
parameter
weight
bias
activation function

ترکیب فارسی و انگلیسی نیز در داده‌های واقعی بسیار رایج است. برای مثال ممکن است یک پژوهشگر بنویسد: «مدل Transformer با استفاده از مکانیزم Self-Attention می‌تواند وابستگی‌های موجود در sequence را یاد بگیرد.» همچنین ممکن است جمله‌ای مانند «learning rate برابر با 3×10^-4 انتخاب شد» در یک مقاله فارسی ظاهر شود.

در برنامه‌نویسی نیز ترکیب زبان‌ها بسیار رایج است؛ برای مثال:

learning_rate = 0.0003  # نرخ یادگیری
batch_size = 32         # اندازه batch
seq_len = 128           # طول توالی

اگر loss_train از 5.2 به 4.8 و سپس به 4.3 کاهش پیدا کند، می‌توان گفت خطای آموزش در حال کاهش است. اما اگر loss_valid ابتدا از 5.5 به 4.9 کاهش پیدا کند و سپس دوباره به 5.1 افزایش یابد، ممکن است مدل نسبت به داده‌های آموزش بیش از حد سازگار شده باشد.

اعداد و واحدها نیز باید به‌درستی پردازش شوند:

0
1
-1
3.14159
2.71828
0.0003
1e-5
99.99%
2026
10,000
2.99M
60M
4B
128
1024
3 × 10^8
9.81 m/s²
70 kg
25 °C
220 V
5 A

علائم نگارشی و نمادهای مختلف:

. , ; : ! ? ( ) [ ] { } " ' ` ~ @ # $ % ^ & * + - = / \ | < > _

نشانی‌های متداول در اینترنت نیز ممکن است در داده وجود داشته باشند:

https://example.com
https://example.com/api/v1/model
user@example.com
github.com/example/project
www.example.org

یک JSON نمونه:

{
  "model": "ark-alpha-mini-v1",
  "vocab_size": 10000,
  "n_embd": 64,
  "n_layer": 4,
  "n_head": 4,
  "learning_rate": 0.0003
}

یک YAML نمونه:

model:
  name: ark-alpha-mini-v1
  vocab_size: 10000
  embedding_size: 64

training:
  batch_size: 32
  seq_len: 128
  learning_rate: 3e-4

در Markdown نیز ممکن است داده‌هایی مانند این وجود داشته باشد:

# Artificial Intelligence

## Machine Learning

- Data
- Model
- Loss
- Optimizer
- Parameters

**Transformer** is a neural network architecture.

`loss = cross_entropy(logits, targets)`

حتی یک متن علمی می‌تواند ترکیبی از فارسی، انگلیسی، ریاضی و نمادهای تخصصی باشد:

اگر یک مدل دارای N پارامتر باشد و با D توکن آموزش داده شود، در یک تقریب ساده می‌توان نسبت داده به پارامتر را به شکل D/N تعریف کرد. برای نمونه اگر N = 3,000,000 و D = 60,000,000 باشد، داریم:

D / N = 60,000,000 / 3,000,000 = 20

این نسبت به‌تنهایی کیفیت آموزش را تعیین نمی‌کند و عواملی مانند کیفیت داده، تکرارپذیری داده، تنوع حوزه‌ها، معماری مدل، tokenizer، optimizer، learning rate و تعداد epochها نیز اهمیت دارند.

در نهایت، یک دیتاست واقعی ممکن است شامل موضوعات بسیار متفاوتی باشد: ادبیات، تاریخ، جغرافیا، پزشکی، اقتصاد، فیزیک، شیمی، ریاضیات، برنامه‌نویسی، مهندسی، هوش مصنوعی، زبان انگلیسی، فلسفه، علوم اجتماعی و مطالب عمومی. بنابراین tokenizer باید بتواند هم واژه‌های رایج فارسی و هم اصطلاحات تخصصی، نام‌های خاص، اعداد، واحدها، فرمول‌ها، کدهای برنامه‌نویسی، URLها، ایمیل‌ها و متن‌های ترکیبی را به شکل مناسبی نمایش دهد."""

enc = test_tokenizer.encode(
    target
)

decoded = test_tokenizer.decode(
    enc.ids
)

print("=" * 100)
print("📚 FULL TARGET TOKENIZER TEST")
print("=" * 100)

print("\nOriginal:")
print(target)

print("\nDecoded:")
print(decoded)

print(
    "\nMATCH:",
    target == decoded
)

print(
    "Token count:",
    len(enc.ids)
)

📚 FULL TARGET TOKENIZER TEST

Original:
هوش مصنوعی یکی از مهم‌ترین شاخه‌های علوم کامپیوتر است که هدف آن ساخت سیستم‌هایی است که بتوانند وظایفی مانند یادگیری، استدلال، تشخیص الگو، پردازش زبان طبیعی و تصمیم‌گیری را انجام دهند. یادگیری ماشین یا Machine Learning زیرمجموعه‌ای از هوش مصنوعی است که در آن مدل به جای دریافت مجموعه‌ای از قوانین ثابت، الگوها را از داده‌ها یاد می‌گیرد.

برای مثال، یک مدل زبانی می‌تواند جمله‌ای مانند «امروز هوا بسیار خوب است» را به مجموعه‌ای از tokenها تبدیل کند. سپس هر token به یک شناسه عددی تبدیل می‌شود و مدل این شناسه‌ها را به embedding vector تبدیل می‌کند. اگر vocabulary شامل 10,000 token باشد، هر token یک شناسه در بازه 0 تا 9,999 خواهد داشت.

یکی از معماری‌های معروف برای مدل‌های زبانی Transformer است. در این معماری، مکانیزم Attention به مدل اجازه می‌دهد ارتباط میان بخش‌های مختلف یک توالی را بررسی کند. برای مثال، در جمله «علی کتابی را که دیروز خریده بود خواند»، مدل باید بتواند رابطه میان «کتابی» و «خریده بود» را تا حدی درک کند.

فرمول ساده Attention به صورت زیر 

In [75]:
print("=" * 100)
print("TOKENIZER ROUND-TRIP TEST")
print("=" * 100)

enc = test_tokenizer.encode(target)
decoded = test_tokenizer.decode(enc.ids)

print("Original length :", len(target))
print("Decoded length  :", len(decoded))
print("Token count     :", len(enc.ids))
print("Exact match     :", target == decoded)

if target != decoded:
    print("\n⚠️ Difference detected!")

    min_len = min(len(target), len(decoded))

    for i in range(min_len):
        if target[i] != decoded[i]:
            print("First difference at position:", i)
            print("Original:", repr(target[i:i+50]))
            print("Decoded :", repr(decoded[i:i+50]))
            break

    if len(target) != len(decoded):
        print("\nLength mismatch!")

TOKENIZER ROUND-TRIP TEST
Original length : 7304
Decoded length  : 7305
Token count     : 2667
Exact match     : False

⚠️ Difference detected!
First difference at position: 0
Original: 'هوش مصنوعی یکی از مهم\u200cترین شاخه\u200cهای علوم کامپیوتر '
Decoded : ' هوش مصنوعی یکی از مهم\u200cترین شاخه\u200cهای علوم کامپیوتر'

Length mismatch!


In [79]:
# ============================================================
# ARK TOKENIZER — FINAL VALIDATION SUITE
# ============================================================

# ------------------------------------------------------------
# 1. TEST DATA
# ------------------------------------------------------------

TEST_CASES = {
    "persian": """
هوش مصنوعی یکی از مهم‌ترین شاخه‌های علوم کامپیوتر است.
یادگیری ماشین به مدل‌ها اجازه می‌دهد الگوها را از داده‌ها یاد بگیرند.
مدل زبانی باید بتواند زبان فارسی را به‌درستی پردازش کند.
""",

    "english": """
Artificial intelligence is a branch of computer science.
Machine learning allows models to learn patterns from data.
Transformers use attention mechanisms to process sequences.
""",

    "mixed": """
مدل Transformer با استفاده از Self-Attention می‌تواند
وابستگی‌های موجود در sequence را یاد بگیرد.
The learning rate is 3e-4 and batch_size is 32.
""",

    "numbers": """
0
1
-1
3.14159
2.71828
0.0003
1e-5
99.99%
2026
10,000
2.99M
60M
4B
128
1024
3 × 10^8
9.81 m/s²
70 kg
25 °C
220 V
5 A
""",

    "math": """
x = 5
y = 3
x + y = 8
x × y = 15

a^2 + b^2 = c^2

f(x) = x^2
f'(x) = 2x

∫ x^2 dx = x^3/3 + C

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
""",

    "code": '''
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

for i in range(10):
    print(fibonacci(i))

learning_rate = 3e-4
batch_size = 32
seq_len = 128
is_training = True
''',

    "json": '''
{
  "model": "ark-alpha-mini-v1",
  "vocab_size": 10000,
  "n_embd": 64,
  "n_layer": 4,
  "n_head": 4,
  "learning_rate": 0.0003
}
''',

    "yaml": '''
model:
  name: ark-alpha-mini-v1
  vocab_size: 10000
  embedding_size: 64

training:
  batch_size: 32
  seq_len: 128
  learning_rate: 3e-4
''',

    "markdown": '''
# Artificial Intelligence

## Machine Learning

- Data
- Model
- Loss
- Optimizer
- Parameters

**Transformer** is a neural network architecture.

`loss = cross_entropy(logits, targets)`
''',

    "urls": """
https://example.com
https://example.com/api/v1/model
https://huggingface.co/
github.com/example/project
www.example.org
user@example.com
""",

    "symbols": r'''
. , ; : ! ? ( ) [ ] { } " ' ` ~ @ # $ % ^ & * + - = / \ | < > _
''',

    "unicode": """
سلام دنیا 🌍
هوش مصنوعی 🤖
یادگیری ماشین 🧠
α β γ
Δ Σ Ω
→ ← ↑ ↓
✓ ✗
© ® ™
""",

    "long_persian": """
این یک آزمایش طولانی برای بررسی عملکرد tokenizer در زبان فارسی است.
مدل زبانی باید بتواند جمله‌های طولانی، پاراگراف‌ها، علائم نگارشی،
اعداد، کلمات انگلیسی، اصطلاحات تخصصی و ساختارهای مختلف را پردازش کند.
در یک مدل Transformer، مکانیزم Attention به مدل اجازه می‌دهد ارتباط
میان tokenهای مختلف را بررسی کند و بر اساس context موجود، token بعدی
را پیش‌بینی نماید. هدف این آزمایش بررسی صحت تبدیل متن به token و سپس
بازسازی متن اولیه از روی token IDها است.
"""
}


# ------------------------------------------------------------
# 2. BASIC ROUND-TRIP TEST
# ------------------------------------------------------------

def test_round_trip(tokenizer, name, text):
    enc = tokenizer.encode(text)
    decoded = tokenizer.decode(enc.ids)

    exact = text == decoded

    return {
        "name": name,
        "chars": len(text),
        "tokens": len(enc.ids),
        "exact_match": exact,
        "decoded": decoded,
        "ids": enc.ids,
    }


# ------------------------------------------------------------
# 3. FIRST DIFFERENCE
# ------------------------------------------------------------

def find_first_difference(a, b):
    n = min(len(a), len(b))

    for i in range(n):
        if a[i] != b[i]:
            return {
                "position": i,
                "original": repr(a[i:i+50]),
                "decoded": repr(b[i:i+50]),
            }

    if len(a) != len(b):
        return {
            "position": n,
            "original": repr(a[n:n+50]),
            "decoded": repr(b[n:n+50]),
        }

    return None


# ------------------------------------------------------------
# 4. TOKEN STATISTICS
# ------------------------------------------------------------

def token_statistics(tokenizer, text):
    enc = tokenizer.encode(text)

    token_count = len(enc.ids)
    char_count = len(text)

    if token_count == 0:
        return {
            "tokens": 0,
            "chars": char_count,
            "chars_per_token": 0,
        }

    return {
        "tokens": token_count,
        "chars": char_count,
        "chars_per_token": char_count / token_count,
    }


# ------------------------------------------------------------
# 5. TOKEN FREQUENCY
# ------------------------------------------------------------

def token_frequency(tokenizer, text, top_n=20):
    enc = tokenizer.encode(text)

    counter = Counter(enc.ids)

    return counter.most_common(top_n)


# ------------------------------------------------------------
# 6. SPECIAL CHARACTER TEST
# ------------------------------------------------------------

def special_character_test(tokenizer):
    samples = [
        "سلام",
        "سلام!",
        "سلام؟",
        "سلام،",
        "hello",
        "hello!",
        "123",
        "3.14",
        "3e-4",
        "a_b",
        "a-b",
        "a/b",
        "user@example.com",
        "https://example.com",
        "αβγ",
        "x^2",
        "QK^T",
        "🙂",
    ]

    results = []

    for text in samples:
        enc = tokenizer.encode(text)
        decoded = tokenizer.decode(enc.ids)

        results.append({
            "text": text,
            "ids": enc.ids,
            "decoded": decoded,
            "match": text == decoded,
        })

    return results


# ------------------------------------------------------------
# 7. WHITESPACE TEST
# ------------------------------------------------------------

def whitespace_test(tokenizer):
    samples = [
        "سلام",
        " سلام",
        "سلام ",
        " سلام ",
        "سلام  دنیا",
        "سلام\nدنیا",
        "سلام\n\nدنیا",
        "سلام\tدنیا",
        "  سلام  دنیا  ",
    ]

    results = []

    for text in samples:
        enc = tokenizer.encode(text)
        decoded = tokenizer.decode(enc.ids)

        results.append({
            "original": repr(text),
            "decoded": repr(decoded),
            "match": text == decoded,
            "ids": enc.ids,
        })

    return results


# ------------------------------------------------------------
# 8. REPEATED TEXT TEST
# ------------------------------------------------------------

def repeated_text_test(tokenizer):
    samples = [
        "سلام " * 20,
        "machine learning " * 20,
        "هوش مصنوعی " * 20,
        "1234567890 " * 20,
        "Transformer " * 20,
    ]

    results = []

    for text in samples:
        enc = tokenizer.encode(text)
        decoded = tokenizer.decode(enc.ids)

        results.append({
            "chars": len(text),
            "tokens": len(enc.ids),
            "match": text == decoded,
        })

    return results


# ------------------------------------------------------------
# 9. VOCABULARY TEST
# ------------------------------------------------------------

def vocabulary_test(tokenizer):
    vocab_size = tokenizer.get_vocab_size()

    ids = []

    for token_id in range(vocab_size):
        try:
            token = tokenizer.id_to_token(token_id)

            if token is not None:
                ids.append(token_id)

        except Exception:
            pass

    unique_ids = len(set(ids))

    return {
        "declared_vocab_size": vocab_size,
        "accessible_ids": len(ids),
        "unique_ids": unique_ids,
        "coverage": unique_ids / vocab_size if vocab_size else 0,
    }


# ------------------------------------------------------------
# 10. ID CONSISTENCY TEST
# ------------------------------------------------------------

def id_consistency_test(tokenizer):
    vocab_size = tokenizer.get_vocab_size()

    problems = []

    for token_id in range(vocab_size):
        try:
            token = tokenizer.id_to_token(token_id)

            if token is None:
                continue

            recovered_id = tokenizer.token_to_id(token)

            if recovered_id != token_id:
                problems.append({
                    "id": token_id,
                    "token": repr(token),
                    "recovered_id": recovered_id,
                })

        except Exception as e:
            problems.append({
                "id": token_id,
                "error": str(e),
            })

    return problems


# ------------------------------------------------------------
# 11. RANDOM ROUND-TRIP TEST
# ------------------------------------------------------------

def random_round_trip_test(tokenizer):
    samples = [
        "هوش مصنوعی",
        "Machine Learning",
        "Transformer Attention",
        "مدل ARK",
        "loss_train = 4.25",
        "learning_rate = 3e-4",
        "QK^T / sqrt(d_k)",
        "سلام دنیا!",
        "hello world!",
        "user@example.com",
        "https://example.com/api",
        "2.99M parameters",
        "60M tokens",
        "مدل 2.99M پارامتری",
        "این یک متن ترکیبی است: Transformer + Attention",
    ]

    failures = []

    for text in samples:
        enc = tokenizer.encode(text)
        decoded = tokenizer.decode(enc.ids)

        if text != decoded:
            failures.append({
                "text": repr(text),
                "decoded": repr(decoded),
                "difference": find_first_difference(text, decoded),
            })

    return failures


# ============================================================
# RUN ALL TESTS
# ============================================================

print("=" * 100)
print("🚀 ARK TOKENIZER — FINAL VALIDATION SUITE")
print("=" * 100)

print("\n📌 Vocabulary")
print("-" * 100)

vocab_info = vocabulary_test(test_tokenizer)

for k, v in vocab_info.items():
    if isinstance(v, float):
        print(f"{k:25}: {v:.4f}")
    else:
        print(f"{k:25}: {v}")


# ------------------------------------------------------------
# ROUND TRIP
# ------------------------------------------------------------

print("\n\n📌 ROUND-TRIP TEST")
print("-" * 100)

all_results = []

for name, text in TEST_CASES.items():

    result = test_round_trip(
        test_tokenizer,
        name,
        text
    )

    all_results.append(result)

    status = "✅ PASS" if result["exact_match"] else "❌ FAIL"

    print(
        f"{status:10} "
        f"{name:20} "
        f"chars={result['chars']:6} "
        f"tokens={result['tokens']:6}"
    )

    if not result["exact_match"]:
        diff = find_first_difference(
            text,
            result["decoded"]
        )

        print("   First difference:", diff)


# ------------------------------------------------------------
# TOKEN EFFICIENCY
# ------------------------------------------------------------

print("\n\n📌 TOKEN EFFICIENCY")
print("-" * 100)

ratios = []

for result in all_results:

    if result["tokens"] > 0:

        ratio = result["chars"] / result["tokens"]

        ratios.append(ratio)

        print(
            f"{result['name']:20} "
            f"{ratio:.3f} chars/token"
        )

if ratios:

    print("\nAverage chars/token:",
          f"{statistics.mean(ratios):.3f}")


# ------------------------------------------------------------
# SPECIAL CHARACTERS
# ------------------------------------------------------------

print("\n\n📌 SPECIAL CHARACTER TEST")
print("-" * 100)

special_results = special_character_test(test_tokenizer)

special_failures = 0

for item in special_results:

    status = "✅" if item["match"] else "❌"

    print(
        f"{status} "
        f"{item['text']!r:30} "
        f"tokens={len(item['ids'])}"
    )

    if not item["match"]:
        special_failures += 1


# ------------------------------------------------------------
# WHITESPACE
# ------------------------------------------------------------

print("\n\n📌 WHITESPACE TEST")
print("-" * 100)

whitespace_results = whitespace_test(test_tokenizer)

whitespace_failures = 0

for item in whitespace_results:

    status = "✅" if item["match"] else "❌"

    print(
        f"{status} "
        f"{item['original']}"
        f" → "
        f"{item['decoded']}"
    )

    if not item["match"]:
        whitespace_failures += 1


# ------------------------------------------------------------
# REPEATED TEXT
# ------------------------------------------------------------

print("\n\n📌 REPETITION TEST")
print("-" * 100)

repetition_results = repeated_text_test(test_tokenizer)

for item in repetition_results:

    status = "✅" if item["match"] else "❌"

    print(
        f"{status} "
        f"chars={item['chars']:6} "
        f"tokens={item['tokens']:6}"
    )


# ------------------------------------------------------------
# ID CONSISTENCY
# ------------------------------------------------------------

print("\n\n📌 TOKEN ID CONSISTENCY")
print("-" * 100)

id_problems = id_consistency_test(test_tokenizer)

if not id_problems:

    print("✅ Every tested token has a consistent ID mapping.")

else:

    print(
        f"❌ Found {len(id_problems)} ID consistency problems."
    )

    for problem in id_problems[:20]:
        print(problem)


# ------------------------------------------------------------
# RANDOM / MIXED TEST
# ------------------------------------------------------------

print("\n\n📌 MIXED INPUT TEST")
print("-" * 100)

random_failures = random_round_trip_test(test_tokenizer)

if not random_failures:

    print("✅ All mixed-input tests passed.")

else:

    print(
        f"❌ {len(random_failures)} mixed-input tests failed."
    )

    for failure in random_failures:
        print(failure)


# ============================================================
# FINAL REPORT
# ============================================================

round_trip_failures = sum(
    not r["exact_match"]
    for r in all_results
)

round_trip_total = len(all_results)

print("\n")
print("=" * 100)
print("📊 FINAL TOKENIZER REPORT")
print("=" * 100)

print(
    f"Round-trip tests       : "
    f"{round_trip_total - round_trip_failures}/"
    f"{round_trip_total} passed"
)

print(
    f"Special character tests: "
    f"{len(special_results) - special_failures}/"
    f"{len(special_results)} passed"
)

print(
    f"Whitespace tests       : "
    f"{len(whitespace_results) - whitespace_failures}/"
    f"{len(whitespace_results)} passed"
)

print(
    f"Mixed-input tests      : "
    f"{len(random_failures) == 0}"
)

print(
    f"ID consistency problems : "
    f"{len(id_problems)}"
)

print(
    f"Vocabulary size        : "
    f"{vocab_info['declared_vocab_size']}"
)


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

critical_failure = (
    round_trip_failures > 0
    or
    len(random_failures) > 0
    or
    len(id_problems) > 0
)

print("\n" + "=" * 100)

if critical_failure:

    print("❌ TOKENIZER VALIDATION: REVIEW REQUIRED")

else:

    print("✅ TOKENIZER VALIDATION: PASSED")

print("=" * 100)

🚀 ARK TOKENIZER — FINAL VALIDATION SUITE

📌 Vocabulary
----------------------------------------------------------------------------------------------------
declared_vocab_size      : 10000
accessible_ids           : 10000
unique_ids               : 10000
coverage                 : 1.0000


📌 ROUND-TRIP TEST
----------------------------------------------------------------------------------------------------
❌ FAIL     persian              chars=   183 tokens=    57
   First difference: {'position': 0, 'original': "'\\nهوش مصنوعی یکی از مهم\\u200cترین شاخه\\u200cهای علوم کامپیوتر'", 'decoded': "' \\nهوش مصنوعی یکی از مهم\\u200cترین شاخه\\u200cهای علوم کامپیوت'"}
❌ FAIL     english              chars=   178 tokens=    64
   First difference: {'position': 0, 'original': "'\\nArtificial intelligence is a branch of computer s'", 'decoded': "' \\nArtificial intelligence is a branch of computer '"}
❌ FAIL     mixed                chars=   147 tokens=    50
   First difference: {'position': 0, 

In [80]:
samples = [
    "هوش مصنوعی",
    "یادگیری ماشین",
    "شبکه عصبی مصنوعی",
    "مدل Transformer",
    "Self-Attention",
    "Multi-Head Attention",
    "gradient descent",
    "learning rate",
    "loss_train",
    "loss_valid",
    "2.99M parameters",
    "60M tokens",
    "QK^T / sqrt(d_k)",
    "مدل 2.99M پارامتری",
    "هوش مصنوعی یکی از مهم‌ترین شاخه‌های علوم کامپیوتر است.",
]

for text in samples:

    enc = tokenizer.encode(text)

    print("=" * 80)
    print("TEXT:")
    print(repr(text))

    print("\nIDS:")
    print(enc.ids)

    print("\nTOKENS:")
    print([repr(tokenizer.id_to_token(i)) for i in enc.ids])

    print("\nTOKEN COUNT:")
    print(len(enc.ids))

    print("\nCHARS/TOKEN:")
    print(len(text) / len(enc.ids))

    print("\nDECODE:")
    print(repr(tokenizer.decode(enc.ids)))

TEXT:
'هوش مصنوعی'

IDS:
[464, 534]

TOKENS:
["'ĠÙĩÙĪØ´'", "'ĠÙħØµÙĨÙĪØ¹ÛĮ'"]

TOKEN COUNT:
2

CHARS/TOKEN:
5.0

DECODE:
' هوش مصنوعی'
TEXT:
'یادگیری ماشین'

IDS:
[681, 806]

TOKENS:
["'ĠÛĮØ§Ø¯Ú¯ÛĮØ±ÛĮ'", "'ĠÙħØ§Ø´ÛĮÙĨ'"]

TOKEN COUNT:
2

CHARS/TOKEN:
6.5

DECODE:
' یادگیری ماشین'
TEXT:
'شبکه عصبی مصنوعی'

IDS:
[624, 827, 534]

TOKENS:
["'ĠØ´Ø¨Ú©Ùĩ'", "'ĠØ¹ØµØ¨ÛĮ'", "'ĠÙħØµÙĨÙĪØ¹ÛĮ'"]

TOKEN COUNT:
3

CHARS/TOKEN:
5.333333333333333

DECODE:
' شبکه عصبی مصنوعی'
TEXT:
'مدل Transformer'

IDS:
[482, 9530]

TOKENS:
["'ĠÙħØ¯ÙĦ'", "'ĠTransformer'"]

TOKEN COUNT:
2

CHARS/TOKEN:
7.5

DECODE:
' مدل Transformer'
TEXT:
'Self-Attention'

IDS:
[6388, 14, 5841]

TOKENS:
["'ĠSelf'", "'-'", "'Attention'"]

TOKEN COUNT:
3

CHARS/TOKEN:
4.666666666666667

DECODE:
' Self-Attention'
TEXT:
'Multi-Head Attention'

IDS:
[792, 5440, 14, 5867, 974, 8465]

TOKENS:
["'ĠM'", "'ulti'", "'-'", "'He'", "'ad'", "'ĠAttention'"]

TOKEN COUNT:
6

CHARS/TOKEN:
3.3333333333333335

DECODE:
' Multi-Head Attention'
TEXT:
'gr